In [1]:
!sudo apt-get update -y
!sudo apt-get install -y curl wget zstd
!curl -fsSL https://ollama.com/install.sh | sh

Get:1 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:2 https://cli.github.com/packages stable InRelease [3,917 B]               
Get:3 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:4 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease [1,578 B]
Hit:5 http://archive.ubuntu.com/ubuntu jammy InRelease                         
Get:6 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease [18.1 kB]
Hit:7 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Hit:8 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease    
Get:9 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]        
Get:10 http://security.ubuntu.com/ubuntu jammy-security/multiverse amd64 Packages [84.6 kB]
Get:11 http://security.ubuntu.com/ubuntu jammy-security/restricted amd64 Packages [7,843 kB]
Get:12 https://cli.github.com/packages stable/main amd64 Package

In [2]:
!wget https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb -O /tmp/cloudflared.deb
!sudo dpkg -i /tmp/cloudflared.deb

--2026-10-02 00:50:37--  https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
Resolving github.com (github.com)... 140.82.121.4
Connecting to github.com (github.com)|140.82.121.4|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://github.com/cloudflare/cloudflared/releases/download/2026.9.3/cloudflared-linux-amd64.deb [following]
--2026-10-02 00:50:37--  https://github.com/cloudflare/cloudflared/releases/download/2026.9.3/cloudflared-linux-amd64.deb
Reusing existing connection to github.com:443.
HTTP request sent, awaiting response... 302 Found
Location: https://release-assets.githubusercontent.com/github-production-release-asset/106867604/edb34d12-c0f8-4ce8-9d92-ccdb5e93b0eb?sp=r&sv=2018-11-09&sr=b&spr=https&se=2026-10-02T01%3A49%3A45Z&rscd=attachment%3B+filename%3Dcloudflared-linux-amd64.deb&rsct=application%2Foctet-stream&skoid=96c2d410-5711-43a1-aedd-ab1947aa7ab0&sktid=398a6654-997b-47e9-b12b-9515b896b4de&

creando el servidor de Ollama

In [18]:
import subprocess
import os
import time

os.environ["OLLAMA_CONTEXT_LENGTH"] = "65536"
os.environ["OLLAMA_FLASH_ATTENTION"] = "1"
os.environ["OLLAMA_KEEP_ALIVE"] = "-1"
os.environ["OLLAMA_KV_CACHE_TYPE"] = "q8_0"
os.environ["OLLAMA_DEBUG"] = "1"

ollama_log = open("/tmp/ollama.log", "w")

ollama_process = subprocess.Popen(
    ["ollama", "serve"],
    stdout=ollama_log,
    #stdout=open("/tmp/ollama.log", "w"),
    stderr=subprocess.STDOUT,
    env=os.environ.copy()
)
time.sleep(5)

print("Ollama iniciado")
print("PID:", ollama_process.pid)

Ollama iniciado con contexto de 64K
PID: 1480


In [19]:
!ollama --version
!cloudflared --version

]11;?\ollama version is 0.35.0
cloudflared version 2026.9.3 (built 2026-09-24-16:07 UTC)


In [20]:
!ollama run hf.co/HauhauCS/Gemma-4-E4B-Uncensored-HauhauCS-Aggressive:Q4_K_M "say Hi!when ready"

]11;?\⠙ ⠙ ⠸ ⠸ ⠴ ⠦ ⠧ ⠧ ⠇ ⠋ ⠙ ⠙ ⠸ ⠸ ⠴ ⠦ ⠦ ⠧ ⠏ ⠏ ⠙ ⠹ ⠸ ⠼ ⠴ ⠦ ⠧ ⠧ ⠇ ⠏ ⠙ ⠙ ⠹ ⠸ ⠼ ⠦ ⠧ ⠧ ⠏ ⠏ ⠋ ⠙ ⠹ ⠸ ⠼ ⠴ ⠧ ⠧ ⠇ ⠏ ⠙ ⠙ ⠸ ⠸ ⠼ ⠴ ⠧ ⠧ ⠇ ⠏ ⠙ ⠙ ⠹ ⠼ ⠼ ⠦ ⠦ ⠇ ⠏ ⠋ ⠋ ⠙ ⠸ ⠸ ⠴ ⠦ ⠧ ⠧ ⠏ ⠋ ⠙ ⠹ ⠸ ⠸ <|channel>thought
The user is instructing me to say "Hi!" when I am ready. Since I am an AI, 
I am ready immediately. Therefore, the most direct and appropriate response
response is to just say "Hi!".<channel|>Hi! 👋



Cargando el Token de los Secretos para un tunnel fijo de cloudflared

In [21]:
from kaggle_secrets import UserSecretsClient
import os

user_secrets = UserSecretsClient()
secret_value_0 = user_secrets.get_secret("CF_TUNNEL_TOKEN")

os.environ["CF_TUNNEL_TOKEN"] = secret_value_0

print("Token cargado correctamente")

Token cargado correctamente


Creando el Tunnel de Cloudflared

In [22]:
import subprocess
import os
import time

cloudflared_log = open("/tmp/cloudflared.log", "w")

cloudflared_process = subprocess.Popen(
    [
        "cloudflared",
        "tunnel",
        "--no-autoupdate",
        "run",
        "--token",
        os.environ["CF_TUNNEL_TOKEN"]
    ],
   # stdout=open("/tmp/cloudflared.log", "w"),
    stdout=cloudflared_log,
    stderr=subprocess.STDOUT
)

time.sleep(5)

print("Cloudflare Tunnel iniciado")
print("PID:", cloudflared_process.pid)

Cloudflare Tunnel iniciado
PID: 1546


In [8]:
!cat /tmp/cloudflared.log

2026-10-02T01:00:40Z INF Starting tunnel tunnelID=7640291e-0886-43f8-95e2-95c13d03d3f1
2026-10-02T01:00:40Z INF Version 2026.9.3 (Checksum 77e26d8d900e0b8469f416239d14b5f296525fdf79fee6f511ef55609e3fbac2)
2026-10-02T01:00:40Z INF GOOS: linux, GOVersion: go1.26.8, GoArch: amd64
2026-10-02T01:00:40Z INF Settings: map[no-autoupdate:true token:*****]
2026-10-02T01:00:40Z INF cloudflared will not automatically update if installed by a package manager.
2026-10-02T01:00:40Z INF Generated Connector ID: 6d0d1769-f28a-44f8-a770-8da4981421d9
2026-10-02T01:00:40Z INF Initial protocol quic
2026-10-02T01:00:40Z INF ICMP proxy will use 172.19.2.2 as source for IPv4
2026-10-02T01:00:40Z INF ICMP proxy will use ::1 in zone lo as source for IPv6
2026-10-02T01:00:40Z INF ICMP proxy will use 172.19.2.2 as source for IPv4
2026-10-02T01:00:40Z INF ICMP proxy will use ::1 in zone lo as source for IPv6
2026-10-02T01:00:40Z INF Starting metrics server on 127.0.0.1:20241/metrics
2026-10-02T01:00:40Z INF Tunnel 

In [23]:
!curl http://127.0.0.1:11434/api/tags

{"models":[{"name":"hf.co/HauhauCS/Gemma-4-E4B-Uncensored-HauhauCS-Aggressive:Q4_K_M","model":"hf.co/HauhauCS/Gemma-4-E4B-Uncensored-HauhauCS-Aggressive:Q4_K_M","modified_at":"2026-10-02T00:52:00.027576197Z","size":6325575400,"digest":"e147e30d920477f2ec508e7213b3ebf3a0f42a01db018c5ee64d4029e0eeb7d2","details":{"parent_model":"","format":"gguf","family":"gemma4","families":["gemma4"],"parameter_size":"7.52B","quantization_level":"Q4_K_M","context_length":131072,"embedding_length":2560},"capabilities":["tools","completion","vision","audio"]}]}

In [24]:
!curl https://ollama-gpu.jcdcruz.com/api/tags

{"models":[{"name":"hf.co/HauhauCS/Gemma-4-E4B-Uncensored-HauhauCS-Aggressive:Q4_K_M","model":"hf.co/HauhauCS/Gemma-4-E4B-Uncensored-HauhauCS-Aggressive:Q4_K_M","modified_at":"2026-10-02T00:52:00.027576197Z","size":6325575400,"digest":"e147e30d920477f2ec508e7213b3ebf3a0f42a01db018c5ee64d4029e0eeb7d2","details":{"parent_model":"","format":"gguf","family":"gemma4","families":["gemma4"],"parameter_size":"7.52B","quantization_level":"Q4_K_M","context_length":131072,"embedding_length":2560},"capabilities":["tools","completion","vision","audio"]}]}

In [29]:
!ollama ps
!nvidia-smi

]11;?\NAME                                                                ID              SIZE      PROCESSOR    CONTEXT    UNTIL   
hf.co/HauhauCS/Gemma-4-E4B-Uncensored-HauhauCS-Aggressive:Q4_K_M    e147e30d9204    3.5 GB    100% GPU     65536      Forever    
Fri Oct  2 01:28:24 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.178.04             Driver Version: 580.178.04     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:0

Comprobando el Tunnel con variante Antropic

In [26]:
import requests

url = "https://ollama-gpu.jcdcruz.com/v1/messages"

headers = {
    "Content-Type": "application/json",
    "x-api-key": "ollama",
    "anthropic-version": "2023-06-01",
}

payload = {
    "model": "hf.co/HauhauCS/Gemma-4-E4B-Uncensored-HauhauCS-Aggressive:Q4_K_M",
    "max_tokens": 100,
    "messages": [
        {
            "role": "user",
            "content": "Reply exactly with: ANTHROPIC_OK"
        }
    ]
}

response = requests.post(
    url,
    headers=headers,
    json=payload,
    timeout=300
)

print("Status:", response.status_code)
print(response.text)

Status: 200
{"id":"msg_89f44a43f2dd63ba366a1570","type":"message","role":"assistant","model":"hf.co/HauhauCS/Gemma-4-E4B-Uncensored-HauhauCS-Aggressive:Q4_K_M","content":[{"type":"text","text":"ANTHROPIC_OK"}],"stop_reason":"end_turn","usage":{"input_tokens":25,"cache_read_input_tokens":0,"output_tokens":6}}



In [27]:
import requests
import json
import time

payload = {
    "model": "hf.co/HauhauCS/Gemma-4-E4B-Uncensored-HauhauCS-Aggressive:Q4_K_M",
    "messages": [
        {
            "role": "user",
            "content": "Reply only HI"
        }
    ],
    "stream": False,
    "think": False,
    "options": {
        "num_predict": 20
    },
    "keep_alive": -1
}

t0 = time.time()

r = requests.post(
    "http://127.0.0.1:11434/api/chat",
    json=payload,
    timeout=300
)

elapsed = time.time() - t0

data = r.json()

print("HTTP:", r.status_code)
print("Tiempo total:", round(elapsed, 2), "s")
print(json.dumps(data, indent=2))

HTTP: 200
Tiempo total: 0.18 s
{
  "model": "hf.co/HauhauCS/Gemma-4-E4B-Uncensored-HauhauCS-Aggressive:Q4_K_M",
  "created_at": "2026-10-02T01:26:32.052102849Z",
  "message": {
    "role": "assistant",
    "content": "HI"
  },
  "done": true,
  "done_reason": "stop",
  "total_duration": 179499319,
  "load_duration": 1718791,
  "prompt_eval_count": 12,
  "prompt_eval_cached_count": 0,
  "prompt_eval_duration": 143873000,
  "eval_count": 2,
  "eval_duration": 29085000
}


In [32]:
import threading
import time
import datetime
import requests
import subprocess


def kaggle_watchdog():

    while True:

        timestamp = datetime.datetime.now()

        # Comprobar API Ollama
        try:
            r = requests.get(
                "http://127.0.0.1:11434/api/tags",
                timeout=10
            )

            ollama_api = "OK" if r.status_code == 200 else "ERROR"

        except Exception:
            ollama_api = "DOWN"


        # Comprobar procesos reales del sistema
        processes = subprocess.getoutput("ps aux")

        ollama_process = "ollama serve" in processes
        tunnel_process = "cloudflared" in processes


        print(
            timestamp,
            "| Ollama API:",
            ollama_api,
            "| Ollama Process:",
            ollama_process,
            "| Tunnel Process:",
            tunnel_process
        )


        time.sleep(500)


watchdog_thread = threading.Thread(
    target=kaggle_watchdog,
    daemon=True
)

watchdog_thread.start()

print("Kaggle watchdog activo")

Kaggle watchdog activo
2026-10-02 01:33:31.986002 | Ollama API: OK | Ollama Process: True | Tunnel Process: True
2026-10-02 01:33:51.902603 | Ollama API: OK | Ollama Process: True | Tunnel Process: True
2026-10-02 01:34:21.910821 | Ollama API: OK | Ollama Process: True | Tunnel Process: True
2026-10-02 01:34:51.919758 | Ollama API: OK | Ollama Process: True | Tunnel Process: True
2026-10-02 01:35:21.928497 | Ollama API: OK | Ollama Process: True | Tunnel Process: True
